# 集合覆盖 scp41（最小成本）

**问题**：实例来自 OR-Library 的 scp41：m=200 个行元素，n=1000 个列集合。列 j 的成本为 c_j，覆盖的行集合为 S_j（a_ij=1 表示列 j 覆盖行 i）。目标是选择一组列，使每一行至少被一个选中列覆盖，同时总成本最小。

**数学模型**

$$\min \sum_{j=1}^{n} c_j x_j$$

$$\text{s.t.}\quad \sum_{j: i \in S_j} x_j \ge 1,\quad i=1,\dots,m$$

$$x_j\in\{0,1\},\quad j=1,\dots,n$$

数据文件：\`/mnt/d/exactTest/column-generation-testcases/set_covering/scp41.txt\`。文献最优值 429（本套件用直接 MIP 自证）。

## 方法：拉格朗日松弛 + 次梯度

**松弛**：松弛全部行覆盖约束，乘子 π_i ≥ 0。拉格朗日函数

$$L(\pi)=\sum_i \pi_i + \sum_j \min\bigl(0, c_j-\sum_i \pi_i a_{ij}\bigr)$$

因为 x_j∈{0,1} 可逐列独立最小化：当 reduced cost $c_j-\sum_i \pi_i a_{ij}<0$ 时取 x_j=1，否则取 0。对任意 π≥0，L(π) 是原问题下界。

**次梯度与步长**

$$g_i = 1 - \sum_j a_{ij} x_j(\pi),\quad \pi_i \leftarrow \max(0, \pi_i + t g_i),\quad t = \lambda\frac{UB-L(\pi)}{\|g\|^2}$$

λ 初始 2.0，连续 60 次未改进下界则减半；共 600 次迭代。

**修复**

1. 贪心覆盖（ratio greedy）：每次选 cost/newly-covered 最小的列补齐未覆盖行；同时给出 reduced-cost 贪心修复作对照。
2. 最后用 HIGHS 在完整池上解整数 MIP 修复。

**原理要点**

1. 松弛覆盖约束后子问题按列独立，闭式解为 x_j=1 当且仅当 reduced cost<0。
2. 次梯度法求对偶下界；UB 用贪心可行解，随迭代改进。
3. 对偶间隙 = (整数最优 − best_L)/整数最优。
4. 停机：600 次迭代（200~1000 区间内）、总墙钟 110s。
5. 随机性：无随机，全部确定性。

In [1]:
import platform, time, datetime, math, ortools
from ortools.math_opt.python import mathopt

print("python", platform.python_version(), "| ortools", ortools.__version__)

DATA = "/mnt/d/exactTest/column-generation-testcases/set_covering/scp41.txt"
toks = open(DATA).read().split()
m, n = map(int, toks[:2])
costs = list(map(int, toks[2:2+n]))
idx = 2 + n
rows = []
for _ in range(m):
    k = int(toks[idx]); idx += 1
    rows.append([int(t)-1 for t in toks[idx:idx+k]]); idx += k
assert idx == len(toks)
colrows = [[] for _ in range(n)]
for i, row in enumerate(rows):
    for j in row:
        colrows[j].append(i)
print("m,n =", m, n, "| rows parsed =", len(rows), "| tokens consumed =", idx)


python 3.10.20 | ortools 9.15.6755
m,n = 200 1000 | rows parsed = 200 | tokens consumed = 5211


In [2]:
def ratio_greedy(selected):
    sel = set(selected)
    covered = [False]*m
    for j in sel:
        for i in colrows[j]:
            covered[i] = True
    while not all(covered):
        best = None
        for j in range(n):
            if j in sel:
                continue
            new = sum(1 for i in colrows[j] if not covered[i])
            if new == 0:
                continue
            key = costs[j]/new
            if best is None or key < best[0]:
                best = (key, j, new)
        if best is None:
            return None
        j = best[1]
        sel.add(j)
        for i in colrows[j]:
            covered[i] = True
    return list(sel)

def rc_greedy(pi, selected):
    sel = set(selected)
    covered = [False]*m
    for j in sel:
        for i in colrows[j]:
            covered[i] = True
    while not all(covered):
        best = None
        for j in range(n):
            if j in sel:
                continue
            new = 0
            s = 0.0
            for i in colrows[j]:
                if not covered[i]:
                    new += 1
                s += pi[i]
            if new == 0:
                continue
            rc = costs[j] - s
            key = (rc, costs[j], -new)
            if best is None or key < best[0]:
                best = (key, j)
        if best is None:
            return None
        j = best[1]
        sel.add(j)
        for i in colrows[j]:
            covered[i] = True
    return list(sel)

UB = sum(costs[j] for j in ratio_greedy([]))
print("initial ratio greedy UB:", UB)
pi = [0.0]*m
lam = 2.0
best_L = float('-inf')
best_pi = None
best_x = None
no_impr = 0
max_iter = 600
t0 = time.perf_counter()
for it in range(max_iter):
    rc = [0.0]*n
    for j in range(n):
        s = 0.0
        for i in colrows[j]:
            s += pi[i]
        rc[j] = costs[j] - s
    x = [1 if rc[j] < 0 else 0 for j in range(n)]
    L = sum(pi) + sum(min(0.0, rc[j]) for j in range(n))
    if L > best_L + 1e-9:
        best_L = L
        best_pi = pi[:]
        best_x = x[:]
        no_impr = 0
    else:
        no_impr += 1
    g = [1.0]*m
    for j in range(n):
        if x[j]:
            for i in colrows[j]:
                g[i] -= 1.0
    norm2 = sum(gi*gi for gi in g)
    if it % 50 == 0 or it == max_iter-1:
        sel = ratio_greedy([j for j in range(n) if x[j]])
        if sel is not None:
            c = sum(costs[j] for j in sel)
            if c < UB:
                UB = c
    step = 0.0 if norm2 < 1e-12 else lam*(UB-L)/norm2
    for i in range(m):
        pi[i] = max(0.0, pi[i] + step*g[i])
    if no_impr >= 60:
        lam = lam/2.0
        no_impr = 0
wall = time.perf_counter()-t0
sel_ratio = ratio_greedy([j for j in range(n) if best_x[j]])
cost_ratio = sum(costs[j] for j in sel_ratio) if sel_ratio else None
if cost_ratio is not None and cost_ratio < UB:
    UB = cost_ratio
sel_rc = rc_greedy(best_pi, [j for j in range(n) if best_x[j]])
cost_rc = sum(costs[j] for j in sel_rc) if sel_rc else None
print("Lagrangian best lower bound:", best_L)
print("ratio greedy repair cost:", cost_ratio, "| cols:", len(sel_ratio) if sel_ratio else None)
print("reduced-cost greedy repair cost:", cost_rc, "| cols:", len(sel_rc) if sel_rc else None)
print("best feasible UB during subgradient:", UB)
print("subgradient wall:", round(wall, 3), "| iterations:", max_iter, "| final lambda:", lam)

# MIP repair on full pool
t1 = time.perf_counter()
mip = mathopt.Model(name="scp41_lag_repair")
x = [mip.add_variable(lb=0.0, ub=1.0, is_integer=True, name=f"x{j}") for j in range(n)]
mip.minimize_linear_objective(sum(costs[j]*x[j] for j in range(n)))
for i, row in enumerate(rows):
    mip.add_linear_constraint(sum(x[j] for j in row) >= 1.0, name=f"cov{i}")
mres = mathopt.solve(mip, mathopt.SolverType.HIGHS, params=mathopt.SolveParameters(time_limit=datetime.timedelta(seconds=120), enable_output=False))
mv = mres.variable_values(x)
selm = [j for j in range(n) if mv[j] > 0.5]
mobj = mres.objective_value()
print("MIP repair:", mres.termination.reason, "| obj:", mobj, "| best_bound:", mres.best_objective_bound(), "| cols:", len(selm), "| wall:", round(time.perf_counter()-t1, 3))
print("dual gap vs MIP repair:", round((mobj-best_L)/mobj, 6))
print("selected_mip:", sorted(selm))


initial ratio greedy UB: 463


Lagrangian best lower bound: 428.98847764989546
ratio greedy repair cost: 434 | cols: 67
reduced-cost greedy repair cost: 434 | cols: 67
best feasible UB during subgradient: 434
subgradient wall: 1.383 | iterations: 600 | final lambda: 0.125
MIP repair: TerminationReason.OPTIMAL | obj: 429.0 | best_bound: 429.0 | cols: 66 | wall: 0.158
dual gap vs MIP repair: 2.7e-05
selected_mip: [0, 1, 2, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 24, 25, 27, 28, 42, 43, 45, 46, 47, 48, 49, 51, 53, 57, 58, 61, 62, 65, 68, 69, 70, 74, 76, 77, 80, 84, 85, 88, 90, 93, 102, 106, 115, 119, 120, 121, 123, 128, 137, 142, 143, 145, 152, 193, 274, 432]


## 运行结果与结论

上方输出显示：600 次次梯度迭代得到拉格朗日下界 **≈428.9885**；贪心修复给出可行上界 434；完整池 MIP 修复得到并证明最优 **429.0**。对偶间隙（相对 MIP 修复值）≈ **2.7e-05**（约 0.0027%）。

**基准最优值来源**：直接 MIP（01_direct）证明最优值 429.0；拉格朗日下界 428.9885 与整数上界 429.0 的间隙极小。

## 结论

SCP 松弛覆盖约束后按列完全可分离，拉格朗日下界非常接近整数最优；次梯度法简单稳定，是 SCP 下界计算的自然选择。